<a href="https://colab.research.google.com/github/WiebkePetersen/GermevalFlausch/blob/main/flausch_task2_train_token_classifier.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [5]:
# Important for Colab
import os
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

# Flausch detection: Task 2 span classification -- Train BERT Token Classifier  

### Dataset preparation

In [7]:
import pandas as pd
import os
import sys
import numpy as np

# work with the train data only
path = ""
#path = "Input/Data/train/"
data = pd.read_json(path + "train_task1.json")
data2 = pd.read_json(path + "train_task2.json")


In [8]:
# set checkpoint and load tokenizer
from transformers import AutoTokenizer
checkpoint = "deepset/gbert-large"
model_name = checkpoint.split("/")[-1]
tokenizer = AutoTokenizer.from_pretrained(checkpoint)


In [9]:
# filter by flausch=yes for task 2
#mode = "only_flausch"
mode = "all"
if mode =="only_flausch":
    dataflausch = data[data["flausch"]=="yes"].copy().reset_index(drop=True)
if mode =="all":
    dataflausch = data.copy().reset_index(drop=True)

In [11]:
# prüfe wie gut unser test/train split ist:
all1 = pd.read_csv(path + "task1.csv")
all2 = pd.read_csv(path + "task2.csv")
allflausch = all1[all1["flausch"]=="yes"]
print("ratio flausch in all data", allflausch.shape[0]/all1.shape[0])
print("ratio flausch in train data", dataflausch.shape[0]/data.shape[0])
print("flausch / #spans in all data", allflausch.shape[0]/all2.shape[0])
print("flausch / #spans in train data", dataflausch.shape[0]/data2.shape[0])
print("similar distributions")



ratio flausch in all data 0.2907143049896106
ratio flausch in train data 1.0
flausch / #spans in all data 0.6818785999113868
flausch / #spans in train data 2.335177146057975
similar distributions


In [12]:
# generate BIO tag scheme (begin/inside/outside)
# dictionaries to map between tag ids and tag names
tags = sorted(data2["type"].unique())
tag2id = {'O': 0}
for i, tag in enumerate(tags):
    tag2id["I-"+ tag] = 2*i +1
    tag2id["B-"+ tag] = 2*(i+1)

id2tag = {v: k for k, v in tag2id.items()}


In [13]:
#some testing
tokenized = tokenizer(["Das ist ein Horrorfilm!"], return_offsets_mapping=True)
print(tokenized.encodings)
print(tokenized.encodings[0].tokens)
print(tokenized.encodings[0].ids)
print(tokenized.encodings[0].offsets)


[Encoding(num_tokens=8, attributes=[ids, type_ids, tokens, offsets, attention_mask, special_tokens_mask, overflowing])]
['[CLS]', 'Das', 'ist', 'ein', 'Horror', '##film', '!', '[SEP]']
[102, 347, 215, 143, 28816, 5241, 3330, 103]
[(0, 0), (0, 3), (4, 7), (8, 11), (12, 18), (18, 22), (22, 23), (0, 0)]


In [14]:
# main function that creates target token sequences for tokenized training data

def align_labels_with_tokens(tokenizer, data, tag2id):
    # tokenize text and get offset_mapping
    text = data["comment"]
    types = data["types"]  # list of types of the flausch spans
    span_pairs = data["span_pairs"]  # list of boundaries of labeled flausch spans
    tokenized_input = tokenizer(text, return_offsets_mapping=True, truncation=True, max_length=512)
    # encodings[0] für den einzelnen Text
    encoding = tokenized_input.encodings[0]

    token_ids = encoding.ids  # Token-IDs
    label_ids = [tag2id['O'] for i in  range(len(encoding.tokens))]  # Initialisiere mit O-Labels für jedes Wort/Token


    # Iteriere über jeden Span, der gelabelt werden soll
    for i in range(len(span_pairs)):
        span_start_char = span_pairs[i][0]
        span_end_char = span_pairs[i][1]
        span_type = types[i]

        # Finde die Token-Indizes, die diesen Span abdecken
        token_start_idx = encoding.char_to_token(span_start_char)
        token_end_idx = encoding.char_to_token(span_end_char - 1) # end_char ist exklusiv

        # Wenn der Span nicht durch die Tokenisierung abgedeckt wird (z.B. wegen Truncation)
        if token_start_idx is None or token_end_idx is None:
            continue

        # Wenn der Span über mehrere Tokens geht oder ein einzelnes Token ist
        for current_token_idx in range(token_start_idx, token_end_idx + 1):
            if current_token_idx == token_start_idx:
                # Dies ist das erste Token, das den Span abdeckt -> B-Tag
                label_ids[current_token_idx] = tag2id["B-"+span_type]
            else:
                # Alle nachfolgenden Tokens, die denselben Span abdecken -> I-Tag
                # Alle Tokens vom Start bis Ende des Spans sind entweder B oder I.
                # Das erste ist B, der Rest I.

                # Wenn token_start_idx == token_end_idx (einzelnes Token für Span), ist es B.
                # Wenn token_start_idx != token_end_idx, dann erstes ist B, der Rest ist I.
                label_ids[current_token_idx] = tag2id["I-"+ span_type]

    return tokenized_input["input_ids"], label_ids



In [15]:
i=4
input_ids, label_ids = align_labels_with_tokens(tokenizer, dataflausch.loc[i], tag2id)
ex_spans = dataflausch["spans"][i]

print(ex_spans)
for idx,j in enumerate(label_ids):
    if j != 0:
        print(id2tag[j], tokenizer.decode(input_ids[idx]))

['wie wunderschön du in diesem video bist 3']
B-compliment wie
I-compliment wunderschön
I-compliment du
I-compliment in
I-compliment diesem
I-compliment v
I-compliment ##ideo
I-compliment bist
I-compliment 3


In [16]:
import pandas as pd
from datasets import Dataset, Features, Value, ClassLabel, Sequence
from sklearn.model_selection import train_test_split # Für den Split in Train/Test



all_input_ids = []
all_labels = []

print("Beginne mit der Vorverarbeitung des DataFrames...")
for idx, row in dataflausch.iterrows():
    input_ids_for_text, labels_for_text = align_labels_with_tokens(tokenizer, row, tag2id)

    # Sammle die Ergebnisse
    all_input_ids.append(input_ids_for_text)
    all_labels.append(labels_for_text)

    if (idx + 1) % 1000 == 0: # Fortschrittsanzeige alle 1000 Texte
        print(f"Verarbeitete {idx + 1}/{len(dataflausch)} Texte.")

print("Alle Texte erfolgreich vorverarbeitet.")
print(f"Anzahl vorbereiteter Beispiele: {len(all_input_ids)}")

max_label_id = max(id2tag.keys())
id2tag_ordered_list = [None] * (max_label_id + 1) # Erstelle eine Liste der richtigen Größe
for k, v in id2tag.items():
    id2tag_ordered_list[k] = v

features = Features({
    'input_ids': Sequence(Value('int32')), # Sequenz von Ganzzahlen für Token-IDs
    'labels': Sequence(ClassLabel(names=id2tag_ordered_list)) # Sequenz von ClassLabels für NER-Tags
})

# Erstelle ein Dictionary, das die Daten enthält
dataset_dict = {
    'input_ids': all_input_ids,
    'labels': all_labels
}

# Erstelle ein Dataset aus dem Dictionary und den definierten Features
prepared_dataset = Dataset.from_dict(dataset_dict, features=features)

print("\nHugging Face Dataset erfolgreich erstellt.")
print(prepared_dataset)
print("\nBeispiel des ersten Eintrags im Dataset:")
print(prepared_dataset[0])

# Überprüfe den ersten Eintrag noch einmal visuell
first_text = dataflausch["comment"].iloc[0] # .iloc[0] für den ersten Eintrag im DF
first_input_ids = prepared_dataset[0]['input_ids']
first_labels_ids = prepared_dataset[0]['labels']

print(f"\nOriginaltext (erster Eintrag): '{first_text}'")
print(f"Token (decoded): {[tokenizer.decode(t) for t in first_input_ids]}")
print(f"Labels (decoded): {[id2tag[l] for l in first_labels_ids]}")


# Teile das Dataset in Trainings- und Devsets auf (85% Training, 15% Development)
# Remember the current dataset is 90% of the original one. 10% test data have been already set aside
split_dataset = prepared_dataset.train_test_split(test_size=0.15, seed=42)
train_dataset = split_dataset['train']
dev_dataset = split_dataset['test']

# create one datasetdict for train, dev and test
from datasets import DatasetDict
dataset_dict = DatasetDict({
    'train': train_dataset,
    'dev': dev_dataset,
})

print(f"\nTrainingsdatensatzgröße: {len(train_dataset)}")
print(f"Entwicklungsdatensatzgröße: {len(dev_dataset)}")



Beginne mit der Vorverarbeitung des DataFrames...
Verarbeitete 1000/33351 Texte.
Verarbeitete 2000/33351 Texte.
Verarbeitete 3000/33351 Texte.
Verarbeitete 4000/33351 Texte.
Verarbeitete 5000/33351 Texte.
Verarbeitete 6000/33351 Texte.
Verarbeitete 7000/33351 Texte.
Verarbeitete 8000/33351 Texte.
Verarbeitete 9000/33351 Texte.
Verarbeitete 10000/33351 Texte.
Verarbeitete 11000/33351 Texte.
Verarbeitete 12000/33351 Texte.
Verarbeitete 13000/33351 Texte.
Verarbeitete 14000/33351 Texte.
Verarbeitete 15000/33351 Texte.
Verarbeitete 16000/33351 Texte.
Verarbeitete 17000/33351 Texte.
Verarbeitete 18000/33351 Texte.
Verarbeitete 19000/33351 Texte.
Verarbeitete 20000/33351 Texte.
Verarbeitete 21000/33351 Texte.
Verarbeitete 22000/33351 Texte.
Verarbeitete 23000/33351 Texte.
Verarbeitete 24000/33351 Texte.
Verarbeitete 25000/33351 Texte.
Verarbeitete 26000/33351 Texte.
Verarbeitete 27000/33351 Texte.
Verarbeitete 28000/33351 Texte.
Verarbeitete 29000/33351 Texte.
Verarbeitete 30000/33351 Texte.

In [17]:
example = dataset_dict["train"][0]
example
pd.DataFrame([tokenizer.decode(x) for x in example["input_ids"]], [id2tag[x] for x in example["labels"]]).T#.rename(columns={0: 'Tokens', 1: 'Tags'})



,O,O,O,O,O,O,O,O,O,O,...,I-compliment,I-compliment,I-compliment,I-compliment,I-compliment,I-compliment,I-compliment,I-compliment,I-compliment,O
0,[CLS],ich,schl,##af,immer,in,viel,zu,großen,(,...,##ideo,so,schön,aus,!,!,*,-,*,[SEP]


In [18]:
dataset_dict["train"].features["labels"].feature.names

['O',
 'I-affection declaration',
 'B-affection declaration',
 'I-agreement',
 'B-agreement',
 'I-ambiguous',
 'B-ambiguous',
 'I-compliment',
 'B-compliment',
 'I-encouragement',
 'B-encouragement',
 'I-gratitude',
 'B-gratitude',
 'I-group membership',
 'B-group membership',
 'I-implicit',
 'B-implicit',
 'I-positive feedback',
 'B-positive feedback',
 'I-sympathy',
 'B-sympathy']

In [19]:
from collections import defaultdict, Counter

retrieved_id2tag_ordered_list = train_dataset.features['labels'].feature.names

# Initialisiere das Dictionary, um die Häufigkeiten zu speichern
split2freqs = defaultdict(Counter)


# Iteriere durch die Splits
for split_name, dataset_split in dataset_dict.items():
    # Iteriere durch jedes Beispiel im aktuellen Dataset-Split
    for example in dataset_split:
        label_ids_for_example = example['labels']
        for label_id in label_ids_for_example:
            # Konvertiere die numerische Label-ID zurück in den String-Tag
            tag_str = retrieved_id2tag_ordered_list[label_id]

            # Prüfe, ob es sich um einen "B-" Tag handelt
            if tag_str.startswith("B-"):
                # Extrahiere den eigentlichen Entitätstyp (z.B. "ORG" aus "B-ORG")
                tag_type = tag_str.split("-")[1]
                # Inkrementiere den Zähler für diesen Split und diesen Entitätstyp
                split2freqs[split_name][tag_type] += 1

# Erstelle ein Pandas DataFrame aus den gesammelten Häufigkeiten für eine bessere Übersicht
label_frequency_df = pd.DataFrame.from_dict(split2freqs, orient="index").fillna(0).astype(int)

label_frequency_df


,compliment,positive feedback,affection declaration,agreement,gratitude,encouragement,sympathy,implicit,group membership,ambiguous
train,2095,6505,2063,174,182,578,75,139,121,167
dev,403,1080,365,33,29,104,15,33,25,24


In [20]:
# huggingface login
from huggingface_hub import login
login()


In [21]:
# dataset to huggingface

dataset_dict.push_to_hub("Wiebke/FlauschSpans", private=False)

Pushing dataset shards to the dataset hub:   0%|          | 0/1 [00:00<?, ?it/s]

Creating parquet from Arrow format:   0%|          | 0/29 [00:00<?, ?ba/s]

Deleting unused files from dataset repository:   0%|          | 0/1 [00:00<?, ?it/s]

Pushing dataset shards to the dataset hub:   0%|          | 0/1 [00:00<?, ?it/s]

Creating parquet from Arrow format:   0%|          | 0/6 [00:00<?, ?ba/s]

Deleting unused files from dataset repository:   0%|          | 0/1 [00:00<?, ?it/s]

### Evaluation Metric



In [22]:
dataset = dataset_dict

In [23]:
dataset

DatasetDict({
    train: Dataset({
        features: ['input_ids', 'labels'],
        num_rows: 28348
    })
    dev: Dataset({
        features: ['input_ids', 'labels'],
        num_rows: 5003
    })
})

In [24]:
!pip install seqeval
!pip install evaluate

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 3.8 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for seqeval: filename=seqeval-1.2.2-py3-none-any.whl size=16162 sha256=b4d3121cdbaad5f4540ba663d11957565ffacacac80ce0b3d84a4be4ad4a0727
  Stored in directory: /root/.cache/pip/wheels/bc/92/f0/243288f899c2eacdfa8c5f9aede4c71a9bad0ee26a01dc5ead
Successfully built seqeval
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.0/84.0 kB 4.1 MB/s eta 0:00:00


In [25]:
import evaluate
seqeval = evaluate.load("seqeval")

In [26]:
import numpy as np
from seqeval.metrics import f1_score, precision_score, recall_score, classification_report


tags = dataset["train"].features['labels'].feature.names

tag2id = {name: i for i, name in enumerate(tags)}
id2tag = {i: name for i, name in enumerate(tags)}

label_list = list(tag2id.keys())

from seqeval.metrics import f1_score, precision_score, recall_score, classification_report

# Annahme: Diese Variablen sind bereits GLOBAL in deinem Skript definiert:
# tags = dataset["train"].features['labels'].feature.names
# tag2id = {name: i for i, name in enumerate(tags)}
# id2tag = {i: name for i, name in enumerate(tags)}
# label_list = list(tag2id.keys()) # Dies ist die Liste aller deiner String-Labels (z.B. ['O', 'B-affection declaration', ...])

def compute_metrics(p):
    predictions, labels = p
    predictions = np.argmax(predictions, axis=2)

    # Konvertierung der numerischen IDs zurück zu den Label-Strings
    # und Entfernen von Padding-Tokens (-100)
    true_labels = []
    for label_sequence in labels:
        true_labels.append([id2tag[l] for l in label_sequence if l != -100])

    true_predictions = []
    for prediction_sequence, label_sequence in zip(predictions, labels):
        true_predictions.append([
            id2tag[p] for p, l in zip(prediction_sequence, label_sequence) if l != -100
        ])

    # Berechne Metriken mit explizitem zero_division=0
    # Dies ist der Standard von seqeval und sorgt dafür, dass undefinierte Werte 0.0 werden.
    # Es hilft, die UndefinedMetricWarning zu kontrollieren.
    precision = precision_score(true_labels, true_predictions, zero_division=0)
    recall = recall_score(true_labels, true_predictions, zero_division=0)
    f1 = f1_score(true_labels, true_predictions, zero_division=0)

    # Generiere und gib den detaillierten Klassifikationsbericht aus.
    # Das `labels=label_list` Argument stellt sicher, dass alle deine definierten Labels
    # im Bericht erscheinen, auch wenn sie im aktuellen Batch nicht vorkommen.
    report = classification_report(
        true_labels,
        true_predictions,
        digits=4, # Anzahl der Dezimalstellen im Bericht
        zero_division=0, # Steuerung der 0-Division für den Bericht
        #labels=label_list # Wichtig, um alle Labels im Bericht zu haben führt aber zu Versionsproblemen
    )

    print("\n--- Classification Report (Aktueller Evaluationsschritt) ---")
    print(report)
    print("-----------------------------------------------------------")

    return {"precision": precision, "recall": recall, "f1": f1}




In [27]:
from transformers import DataCollatorForTokenClassification
data_collator = DataCollatorForTokenClassification(tokenizer)

### Training

In [28]:
# load model with the correct head for token classification with the number of labels
from transformers import AutoModelForTokenClassification, TrainingArguments, Trainer

num_labels = len(tag2id)  # Anzahl der Labels

model = AutoModelForTokenClassification.from_pretrained(
    checkpoint, num_labels=num_labels, id2label=id2tag, label2id=tag2id
)

model.safetensors:   0%|          | 0.00/1.35G [00:00<?, ?B/s]

Some weights of BertForTokenClassification were not initialized from the model checkpoint at deepset/gbert-large and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [29]:
from transformers import pipeline
i=2
text = tokenizer.decode(dataset["dev"]["input_ids"][i])
classifier = pipeline("ner", model=model, tokenizer=tokenizer)

print(text)
for d in classifier(text):
  print(d["word"], d["entity"])

Device set to use cuda:0


[CLS] Ich gehe morgens gar nicht Duschen [SEP]
[CLS] B-gratitude
Ich B-gratitude
gehe B-gratitude
morgens B-gratitude
gar B-gratitude
nicht B-gratitude
Dusche B-gratitude
##n I-implicit
[SEP] B-gratitude


In [33]:
from transformers import TrainingArguments, Trainer
import os

if mode == "all":
    model_name = model_name + "_all"


# Define training parameters
training_args = TrainingArguments(
    output_dir="flausch_span_"+ model_name, # Verzeichnis für Modell, Checkpoints und Logs
    learning_rate=2e-5,
    metric_for_best_model="eval_f1", # competition ranking metric
    greater_is_better=True, # Setze auf False, wenn du Loss minimieren willst
    per_device_train_batch_size=16, # Reduziere um OOM zu vermeiden
    per_device_eval_batch_size=16,  # Reduziere um OOM zu vermeiden
    num_train_epochs=3,
    weight_decay=0.01,
    eval_strategy="steps",
    save_strategy="steps",
    load_best_model_at_end=True,
    logging_steps=500, # Alle 500 Schritte loggen
    fp16=True, # Für schnellere und speichereffizientere Berechnung auf GPU
    gradient_checkpointing=True, # Hilft gegen OOM bei großen Modellen, macht Training langsamer
    save_total_limit=1, # Nur das beste Modell speichern
    report_to="none",
    push_to_hub=False, # Zuerst trainieren, dann manuell pushen
)

# Define Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=dataset["train"],
    eval_dataset=dataset["dev"],
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics, # Eigene Funktion zur Metrikberechnung
)


<ipython-input-33-2859955319>:30: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


In [34]:
model_name

'gbert-large_all'

In [35]:
print("\nPredictions on dev data before training:")

# Die predict-Methode gibt ein Objekt zurück, das die Vorhersagen,
# die echten Labels und die Metriken enthält (falls der Trainer diese selbst berechnet hat).
predictions_output = trainer.predict(test_dataset=dataset["dev"])



Predictions on dev data before training:



--- Classification Report (Aktueller Evaluationsschritt) ---
                       precision    recall  f1-score   support

affection declaration     0.0000    0.0000    0.0000       366
            agreement     0.0000    0.0000    0.0000        33
            ambiguous     0.0000    0.0000    0.0000        24
           compliment     0.0000    0.0000    0.0000       404
        encouragement     0.0000    0.0000    0.0000       104
            gratitude     0.0000    0.0345    0.0000        29
     group membership     0.0000    0.0000    0.0000        25
             implicit     0.0000    0.0000    0.0000        33
    positive feedback     0.0028    0.0111    0.0044      1080
             sympathy     0.0000    0.0000    0.0000        15

            micro avg     0.0002    0.0062    0.0003      2113
            macro avg     0.0003    0.0046    0.0004      2113
         weighted avg     0.0014    0.0062    0.0023      2113

-----------------------------------------------------

In [36]:

# Training
trainer.train()


Step,Training Loss,Validation Loss,Model Preparation Time,Precision,Recall,F1
500,0.900700,0.797665,0.032800,0.000000,0.000000,0.000000
1000,0.722300,0.621364,0.032800,0.204734,0.151443,0.174102
1500,0.532700,0.404056,0.032800,0.226173,0.358258,0.277289
2000,0.369100,0.334854,0.032800,0.281885,0.433034,0.341482
2500,0.302100,0.301260,0.032800,0.322211,0.452437,0.376378
3000,0.254800,0.265521,0.032800,0.382131,0.526266,0.442763
3500,0.274400,0.266568,0.032800,0.307166,0.403691,0.348875
4000,0.187400,0.280340,0.032800,0.412437,0.546143,0.469965
4500,0.176700,0.262476,0.032800,0.442120,0.580218,0.501842
5000,0.170800,0.262214,0.032800,0.434783,0.582111,0.497774



--- Classification Report (Aktueller Evaluationsschritt) ---
                       precision    recall  f1-score   support

affection declaration     0.0000    0.0000    0.0000       366
            agreement     0.0000    0.0000    0.0000        33
            ambiguous     0.0000    0.0000    0.0000        24
           compliment     0.0000    0.0000    0.0000       404
        encouragement     0.0000    0.0000    0.0000       104
            gratitude     0.0000    0.0000    0.0000        29
     group membership     0.0000    0.0000    0.0000        25
             implicit     0.0000    0.0000    0.0000        33
    positive feedback     0.0000    0.0000    0.0000      1080
             sympathy     0.0000    0.0000    0.0000        15

            micro avg     0.0000    0.0000    0.0000      2113
            macro avg     0.0000    0.0000    0.0000      2113
         weighted avg     0.0000    0.0000    0.0000      2113

-----------------------------------------------------

TrainOutput(global_step=5316, training_loss=0.37622706669447925, metrics={'train_runtime': 2731.2523, 'train_samples_per_second': 31.137, 'train_steps_per_second': 1.946, 'total_flos': 9815926089745128.0, 'train_loss': 0.37622706669447925, 'epoch': 3.0})

In [37]:
text = tokenizer.decode(dataset["dev"]["input_ids"][i])
classifier = pipeline("ner", model=model, tokenizer=tokenizer)

print(text)
for d in classifier(text):
  print(d["word"], d["entity"])

Device set to use cuda:0


[CLS] Ich gehe morgens gar nicht Duschen [SEP]


/usr/local/lib/python3.11/dist-packages/torch/utils/checkpoint.py:87: UserWarning: None of the inputs have requires_grad=True. Gradients will be None
  warnings.warn(


In [38]:

print("\nPredictions on dev set after training.")

# Die predict-Methode gibt ein Objekt zurück, das die Vorhersagen,
# die echten Labels und die Metriken enthält (falls der Trainer diese selbst berechnet hat).
predictions_output = trainer.predict(test_dataset=dataset["dev"])


Predictions on dev set after training.



--- Classification Report (Aktueller Evaluationsschritt) ---
                       precision    recall  f1-score   support

affection declaration     0.4658    0.6694    0.5493       366
            agreement     0.1250    0.0909    0.1053        33
            ambiguous     0.4545    0.4167    0.4348        24
           compliment     0.3047    0.4480    0.3627       404
        encouragement     0.4143    0.5577    0.4754       104
            gratitude     0.5714    0.6897    0.6250        29
     group membership     0.1935    0.2400    0.2143        25
             implicit     0.0000    0.0000    0.0000        33
    positive feedback     0.5054    0.6500    0.5687      1080
             sympathy     0.1111    0.0667    0.0833        15

            micro avg     0.4421    0.5802    0.5018      2113
            macro avg     0.3146    0.3829    0.3419      2113
         weighted avg     0.4357    0.5802    0.4968      2113

-----------------------------------------------------

In [39]:
trainer.push_to_hub()

training_args.bin:   0%|          | 0.00/5.30k [00:00<?, ?B/s]

Upload 2 LFS files:   0%|          | 0/2 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/1.34G [00:00<?, ?B/s]

CommitInfo(commit_url='https://huggingface.co/Wiebke/flausch_span_gbert-large_all/commit/25594dcad6943044047668334cfcba649d8d7def', commit_message='End of training', commit_description='', oid='25594dcad6943044047668334cfcba649d8d7def', pr_url=None, repo_url=RepoUrl('https://huggingface.co/Wiebke/flausch_span_gbert-large_all', endpoint='https://huggingface.co', repo_type='model', repo_id='Wiebke/flausch_span_gbert-large_all'), pr_revision=None, pr_num=None)